In [1]:
%pip install --quiet pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, re, json, unicodedata
import pandas as pd

# === INPUT/OUTPUT PATHS ===
CSV_INPUT = r"C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_extracted_all.csv"
OUT_DIR   = r"C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients"
JSONL_OUT = os.path.join(OUT_DIR, "ingredients_cleaned.jsonl")
JSON_OUT  = os.path.join(OUT_DIR, "ingredients_cleaned_by_product.json")
CSV_OUT   = os.path.join(OUT_DIR, "ingredients_cleaned.csv")

os.makedirs(OUT_DIR, exist_ok=True)

# === Canonical replacements (extend as you like) ===
CANON = {
    "suger": "sugar",
    "sugr": "sugar",
    "flovour": "flavour",
    "flavor": "flavour",      # choose UK/US style you prefer
    "colour": "color",
    "colur": "color",
    "citric accid": "citric acid",
    "wheet": "wheat",
    "maize flour": "corn flour",
    # common symbols
    "&": "and",
    "’": "'",
    "‐": "-", "-": "-", "–": "-", "—": "-",  # various dashes -> hyphen
}

# === Words to drop from ends or standalone (not informative as names) ===
STOP_TOKENS = {
    "and","with","contains","may","or","of","from","to","in","on",
    "natural","artificial","flavourings","flavorings","flavours","flavors",
    "permitted","permitted flavours","permitted flavorings"
}

# === precompiled patterns ===
RE_WHITESPACE  = re.compile(r"\s+")
RE_LIST_SPLIT  = re.compile(r"[;,•·●∙]+")  # split an OCR 'ingredients' field
RE_PARENS      = re.compile(r"\((?:[^()]*|\([^()]*\))*\)")  # remove (…) groups safely
RE_NUM_UNITS   = re.compile(r"(?i)\b\d+(?:[\.,]\d+)?\s*(?:%|g|kg|mg|ml|l|lt|tsp|tbsp|teaspoon|tablespoon|cup|cups)\b")
RE_NUM_STANDALONE = re.compile(r"\b\d+(?:[\.,]\d+)?\b")
RE_E_NUMBERS   = re.compile(r"\bE[-\s]?\d{3,4}\b", flags=re.IGNORECASE)
RE_PUNCT_TRIM  = re.compile(r"^[\s\-\:\.\,\/\+]+|[\s\-\:\.\,\/\+]+$")  # trim ends
RE_MULTI_HYPH  = re.compile(r"-{2,}")
RE_APOSTROPHE_SP = re.compile(r"\s+'\s*")  # spaces around apostrophe
RE_NONWORD     = re.compile(r"[^0-9A-Za-z\-\&\'\s]")  # keep letters, digits, -, &, '

TITLE_EXCEPTIONS = {"of","and","with","in","on","to","for","the","a","an"}

def strip_diacritics(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def canon_replace(s: str) -> str:
    out = s
    for k, v in CANON.items():
        out = out.replace(k, v)
    return out

def norm_whitespace(s: str) -> str:
    return RE_WHITESPACE.sub(" ", s).strip()

def title_case_smart(s: str) -> str:
    words = s.split()
    out = []
    for i, w in enumerate(words):
        lw = w.lower()
        if i > 0 and lw in TITLE_EXCEPTIONS:
            out.append(lw)
        else:
            # preserve Mc/Mac-like? simple title() is fine for now
            out.append(lw.capitalize())
    return " ".join(out)

def clean_one_name(raw: str) -> str:
    """
    Regex-first cleaning to make names human-readable.
    Conservative: removes obvious quantities, %s, parenthetical notes, noise.
    """
    if not raw:
        return ""
    s = str(raw)

    # 1) normalize unicode & lower
    s = strip_diacritics(s)
    s = canon_replace(s)
    s = s.lower()

    # 2) drop parentheses content (often allergens/qty/notes); run twice to be safe on nested OCR
    s = RE_PARENS.sub(" ", s)
    s = RE_PARENS.sub(" ", s)

    # 3) drop quantities like "12 g", "2%", etc., and standalone numbers
    s = RE_NUM_UNITS.sub(" ", s)
    s = RE_NUM_STANDALONE.sub(" ", s)

    # 4) optionally drop E-numbers if you don't want them in names (toggle as you like)
    s = RE_E_NUMBERS.sub(" ", s)

    # 5) remove non-word noise but keep &, -, '
    s = RE_NONWORD.sub(" ", s)

    # 6) normalize whitespace/dashes/apostrophes
    s = RE_MULTI_HYPH.sub("-", s)
    s = RE_APOSTROPHE_SP.sub("'", s)
    s = RE_PUNCT_TRIM.sub("", s)
    s = norm_whitespace(s)

    # 7) simple stop-tokens if the whole thing is just a connector
    if s in STOP_TOKENS:
        return ""

    # 8) canonical after cleanup again (helps "flavor/flavour" etc.)
    s = canon_replace(s)
    s = norm_whitespace(s)

    # 9) final title casing with exceptions
    s = title_case_smart(s)

    return s

def split_ingredient_field(field: str) -> list[str]:
    """
    Split a raw 'ingredients' field into individual entries using robust separators.
    """
    if not field:
        return []
    parts = [p.strip() for p in RE_LIST_SPLIT.split(field) if p.strip()]
    return parts


In [3]:
import re

def _pat(x):
    """Return the pattern string whether x is a compiled regex or a raw string."""
    return x.pattern if hasattr(x, "pattern") else str(x)

def sanitize_regex_pattern(p: str) -> str:
    """
    Remove or neutralize inline flag groups so we can OR-join patterns safely.
    - Turns (?i:...) -> (?:...)   (drop flags, keep non-capturing group)
    - Removes bare globals like (?i), (?m), (?s), (?x), (?a), (?u), (?L)
    """
    # Normalize whitespace
    p = p.strip()

    # 1) Scoped flags like (?i: ... ) -> non-capturing group
    p = re.sub(r'\(\?[aiLmsux]+:', '(?:', p)

    # 2) Bare global flags like (?i) -> remove
    p = re.sub(r'\(\?[aiLmsux]+\)', '', p)

    return p

# Collect your four parts
_parts = [RE_PARENS, RE_NUM_UNITS, RE_NUM_STANDALONE, RE_E_NUMBERS]

safe_parts = []
for i, rx in enumerate(_parts, 1):
    p = _pat(rx)

    # If someone pasted JS-style /pattern/, strip the slashes
    if p.startswith('/') and p.endswith('/'):
        p = p[1:-1]

    p = sanitize_regex_pattern(p)

    # Validate each part individually so you get a clear error if one is bad
    try:
        re.compile(p)
    except re.error as e:
        raise SystemExit(f"Bad regex in part {i}: {e}  ->  {p!r}")

    safe_parts.append(f"(?:{p})")

# Combine with IGNORECASE at compile-time
RE_SPACE_BLOCK = re.compile("|".join(safe_parts), flags=re.IGNORECASE)


In [4]:
import re

def _pat(x):
    return x.pattern if hasattr(x, "pattern") else str(x)

def _sanitize_for_or(p: str) -> str:
    """
    Make a pattern safe to be OR-joined:
    - remove bare inline flags (?i)(?m)(?s)(?x)... anywhere
    - turn scoped flags like (?i:...) into non-capturing (?:...)
    - convert plain capturing groups (...) into non-capturing (?:...)
      (avoids group-number conflicts when joining)
    """
    p = p.strip()
    # strip JS-style /pattern/
    if p.startswith("/") and p.endswith("/"):
        p = p[1:-1]

    # scoped flags (?i:...) -> (?:...)
    p = re.sub(r"\(\?[A-Za-z]*:", "(?:", p)
    # bare flags (?i) -> remove
    p = re.sub(r"\(\?[A-Za-z]*\)", "", p)

    # convert plain capturing groups (...) -> (?:...)
    # (don’t touch special groups like (?P<..>), (?:...), (?=...), (?!...), (?<=...), (?<!...), (?>...))
    p = re.sub(r"\((?!\?)(?!\*)", "(?:", p)

    return p

def build_or_regex(parts, flags=re.IGNORECASE):
    safe_parts = []
    for i, rx in enumerate(parts, 1):
        p = _sanitize_for_or(_pat(rx))
        try:
            re.compile(p)  # validate individually
        except re.error as e:
            raise SystemExit(f"Bad regex in part {i}: {e}  ->  {p!r}")
        safe_parts.append(f"(?:{p})")
    return re.compile("|".join(safe_parts), flags=flags)

# ---- use it here ----
RE_SPACE_BLOCK = build_or_regex([RE_PARENS, RE_NUM_UNITS, RE_NUM_STANDALONE, RE_E_NUMBERS])


In [ ]:
import polars as pl
from pathlib import Path

path = Path(

FileNotFoundError: no workbook found at path 'C:\\path\\to\\ingredients_postprocessed.xlsx'

In [6]:
for name, rx in {
    "RE_PARENS": RE_PARENS,
    "RE_NUM_UNITS": RE_NUM_UNITS,
    "RE_NUM_STANDALONE": RE_NUM_STANDALONE,
    "RE_E_NUMBERS": RE_E_NUMBERS,
}.items():
    p = _sanitize_for_or(_pat(rx))
    try:
        re.compile(p)
        print(f"{name}: OK")
    except re.error as e:
        print(f"{name}: ERROR -> {e} | pattern={p!r}")


RE_PARENS: OK
RE_NUM_UNITS: OK
RE_NUM_STANDALONE: OK
RE_E_NUMBERS: OK


In [15]:
import re, time
import pandas as pd
from pathlib import Path

# (Optional) small speed/memory win on recent pandas
try:
    pd.options.mode.copy_on_write = True
except Exception:
    pass

path = Path(r"C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_postprocessed.xlsx")
out_path = path.with_name(path.stem + "_clean.xlsx")

t0 = time.perf_counter()
df = pd.read_excel(path)  # engine="openpyxl" if needed
t1 = time.perf_counter()

# Auto-detect column (or set COL = "ingredient")
cands = [c for c in df.columns if "ing" in c.lower() or "ingredient" in c.lower()]
COL = cands[0] if cands else df.columns[0]

# Patterns
RE_PARENS         = r"\([^)]*\)"
RE_NUM_UNITS      = r"\b\d+(?:\.\d+)?\s?(?:kg|g|mg|l|ml|oz|lb|tsp|tbsp|cup|cups)\b"
RE_NUM_STANDALONE = r"\b\d+(?:\.\d+)?\b"
RE_E_NUMBERS      = r"\bE\d{3}\b"

# 👉 Single-pass alternation instead of 4 passes
combo_pat = re.compile("|".join([RE_PARENS, RE_NUM_UNITS, RE_NUM_STANDALONE, RE_E_NUMBERS]))

# Use pandas’ vectorized string ops
ing = df[COL].astype("string").fillna("")
ing = ing.str.replace(combo_pat, " ", regex=True) \
         .str.replace(r"\s+", " ", regex=True) \
         .str.strip()

df[f"{COL}_clean"] = ing
t2 = time.perf_counter()

# Excel write is often the slowest part; if you only need a file, CSV is much faster.
# Switch to CSV if possible:
# df.to_csv(path.with_name(path.stem + "_clean.csv"), index=False); t3 = time.perf_counter(); ...

df.to_excel(out_path, index=False)
t3 = time.perf_counter()

print(f"Rows: {len(df):,}")
print(f"Read   time: {t1 - t0:0.3f}s")
print(f"Clean  time: {t2 - t1:0.3f}s")
print(f"Write  time: {t3 - t2:0.3f}s")
print(f"Total  time: {t3 - t0:0.3f}s")
print("Detected ingredient column:", COL)
print(df[[COL, f"{COL}_clean"]].head())


Rows: 1,307
Read   time: 0.232s
Clean  time: 0.011s
Write  time: 0.286s
Total  time: 0.529s
Detected ingredient column: ingredient_raw
                ingredient_raw ingredient_raw_clean
0             2 | Black Pepper       | Black Pepper
1                          NaN                     
2                 ? CT] Potato         ? CT] Potato
3  Acidity Regulator (INS 330)    Acidity Regulator
4                   Bay Leaves           Bay Leaves


In [17]:
import re, time
import pandas as pd
from pathlib import Path

pd.options.mode.copy_on_write = True

path = Path(r"C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_postprocessed.xlsx")
out_path = path.with_name(path.stem + "_clean_and_exploded.xlsx")

t0 = time.perf_counter()
df = pd.read_excel(path)  # engine="openpyxl" if needed
t1 = time.perf_counter()

# Auto-detect the ingredient column
cands = [c for c in df.columns if "ing" in c.lower() or "ingredient" in c.lower()]
COL = cands[0] if cands else df.columns[0]
print("Detected ingredient column:", COL)

# --- Patterns (same logic as before) ---
RE_PARENS         = r"\([^)]*\)"
RE_NUM_UNITS      = r"\b\d+(?:\.\d+)?\s?(?:kg|g|mg|l|ml|oz|lb|tsp|tbsp|cup|cups)\b"
RE_NUM_STANDALONE = r"\b\d+(?:\.\d+)?\b"
RE_E_NUMBERS      = r"\bE\d{3}\b"

combo_pat = re.compile("|".join([RE_PARENS, RE_NUM_UNITS, RE_NUM_STANDALONE, RE_E_NUMBERS]))

# Base clean
ing = df[COL].astype("string").fillna("")
ing = (ing
    .str.replace(combo_pat, " ", regex=True)
    # Normalize common separators to comma
    .str.replace(r"\b(?:and|with)\b", ",", regex=True, case=False)
    .str.replace(r"[|;•·/]+", ",", regex=True)        # pipes, bullets, slashes, etc.
    # Remove stray leading/trailing non-letters, keep hyphens within words
    .str.replace(r"^[^\w]+|[^\w]+$", " ", regex=True)
    # If you want to be stricter (letters only + spaces + hyphen); keep accented Latin letters:
    .str.replace(r"[^A-Za-z\u00C0-\u017F\s\-\,]", " ", regex=True)
    # Collapse whitespace and comma spacing
    .str.replace(r"\s*,\s*", ",", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip(", ").str.strip()
)

df[f"{COL}_clean"] = ing

# --- Explode into one ingredient per row ---
# Split on commas (after we normalized separators to commas), explode, trim again
exploded = (df[[COL, f"{COL}_clean"]]
            .assign(_row_id=lambda x: x.index)  # keep original row mapping
            .assign(item=lambda x: x[f"{COL}_clean"].str.split(r","))
            .explode("item")
            .assign(item=lambda x: x["item"].astype("string").fillna("").str.replace(r"\s+", " ", regex=True).str.strip())
            .query("item != ''")
            .drop_duplicates(subset=["_row_id", "item"])
           )

t2 = time.perf_counter()

# Save two sheets: 1) original+clean, 2) exploded items
with pd.ExcelWriter(out_path, engine="xlsxwriter") as xlw:
    df.to_excel(xlw, sheet_name="cleaned_rows", index=False)
    exploded.rename(columns={
        COL: "ingredient_raw",
        f"{COL}_clean": "ingredient_clean",
        "item": "ingredient_item"
    }).to_excel(xlw, sheet_name="exploded_items", index=False)

t3 = time.perf_counter()

print(f"Rows (original): {len(df):,}")
print(f"Read   time: {t1 - t0:0.3f}s")
print(f"Clean+Explode time: {t2 - t1:0.3f}s")
print(f"Write  time: {t3 - t2:0.3f}s")
print(f"Total  time: {t3 - t0:0.3f}s")
print("Sample exploded:")
print(exploded.head(10))
print(f"\nSaved: {out_path}")


Detected ingredient column: ingredient_raw
Rows (original): 1,307
Read   time: 0.438s
Clean+Explode time: 0.083s
Write  time: 0.655s
Total  time: 1.176s
Sample exploded:
                                      ingredient_raw  \
0                                   2 | Black Pepper   
2                                       ? CT] Potato   
3                        Acidity Regulator (INS 330)   
4                                         Bay Leaves   
5                    Bengat gram \* _| Flour (Besan)   
5                    Bengat gram \* _| Flour (Besan)   
6                                         c / Nutmeg   
6                                         c / Nutmeg   
7                                           Cardamom   
8  Cinnamon and Mint Leaves 4 “| CONTAINS ADDED N...   

                          ingredient_raw_clean  _row_id               item  
0                                 Black Pepper        0       Black Pepper  
2                                    CT Potato        2    